# ***Ab initio* Methods:** Introduction to Single Point, Vibrational modes and Band Structure calculations with PySCF


## Basics of PYSCF for molecular systems

PySCF (Python-based Simulations of Chemistry Framework) is an open-source quantum chemistry package designed for coders, focusing in simplicity and flexibility. Written in Python with efficient C backend components, PySCF allows users to easily build, customize, and automate quantum chemistry calculation. It supports a wide range of methods, including Hartree–Fock (HF), Density Functional Theory (DFT), post-HF methods, and more. It is far from the more efficient code in the literature when we talk about computational performacy, where other codes (like FHI-aims, ORCA, VASP, Quantum Espresso, ...) should be prioritary. PySCF is aimed for development and modularity, or to run small systems.

In this section, we introduce the basic works of PySCF for molecular systems, using a series of hands-on examples to demonstrate how to define a molecule, perform electronic structure calculations, analyze results, and visualize key properties such as molecular orbitals, dipole moments, and electronic spectra.


In [ ]:
!pip --quiet install pyscf ase py3Dmol

In [ ]:
##### A very straightforward exemple with minimal information
from pyscf import gto
from pyscf import scf
import time

#Basic Molecule geometry and proprieties parsing. More features are explained in https://pyscf.org/user/gto.html
mol = gto.M(
    atom = '''
    O  0.000   0.000   0.000
    H  0.000  -0.757   0.587
    H  0.000   0.757   0.587 ''',
    basis = '6-311g')

#Exemple of a basic unrestricted Hartree--Fock calculation.
#mf.kernel returns the total energy, nothing else should be printed with mf.verbose = 0
mf = scf.UHF(mol)
mf.verbose = 0

t0=time.time()
e_RHF = mf.kernel()
tf = time.time()
print(f"Time on the Single Point calculation: {tf-t0} seconds")
print(f"Total HF energy: {e_RHF}")


In [ ]:
##### A similar exemple, now with few keywords added to demonstrate PySCF functionalities
from pyscf import dft,gto,scf

#Molecule Definition
mol = gto.M(
    atom = '''
    O  0.000   0.000   0.000
    H  0.000  -0.757   0.587
    H  0.000   0.757   0.587 ''',
    charge = 0,                  # -1 = one extra electron
    spin = 0,                    # Number of unpared electrons (same as 2S)
    output = 'output.out',       # Direct output of calculation for this molecule to a file
    basis = '6-311g'             # Basis set used on the SCF process
    )


#Same molecule, but now with a different method, restricted DFT (PBE).
mf = scf.RKS(mol).newton()        # .newton() activates second-order self-consistent field (SOSCF)
mf = scf.addons.frac_occ(mf)      # Allow fractional orbital ocupation to help convergence

mf.xc = 'PBE'                     # Functional PBE. Any functional in XCLib can be used
mf.verbose = 10                   # Defines verbosity level. 0 = no output
mf.conv_tol = 1e-10               # Energy convergence criteria, in Eh
mf.max_cycle = 100                # Maximun allowed number of SCF cycles


mf.chkfile = 'chkpoint.dat'       # Checkpoint file to restart calculations
mf.init_guess = 'vsap'            # Strategy used for initial density guess
#mf.init_guess = 'chkpoint.dat'   # Loading old ckeckpoint file


#Runs the SCF
mf.kernel()



In [ ]:
##### Lets get some other physical quantities out of our calculation:
import numpy as np

#Analyze the given SCF object, this is what computes Mulliken population and orbital info
results = mf.analyze()

print("Molecular Properties Computed from PySCF")

# Dipole Moment - 3D vector in Debye. It represents the separation of positive and negative charges.
dipole = mf.dip_moment(unit='Debye')
dipole_magnitude = np.linalg.norm(dipole)
print(f"\nDipole Moment: {dipole_magnitude:.3f} D")
print(f"   Components   : x = {dipole[0]:.3f}, y = {dipole[1]:.3f}, z = {dipole[2]:.3f} (Debye)")

# HOMO-LUMO Gap - The gab in energy between the highest occupied AND the lowest unoccupied molecular orbital.
mo_energy = mf.mo_energy
homo_index = mol.nelectron // 2 - 1
lumo_index = homo_index + 1
homo_energy = mo_energy[homo_index]
lumo_energy = mo_energy[lumo_index]
gap_au = lumo_energy - homo_energy
gap_ev = gap_au * 27.2114  # 1 Hartree = 27.2114 eV
print(f"\nHOMO-LUMO Gap: {gap_ev:.2f} eV ({gap_au:.4f} Hartree)")
print(f"   HOMO Energy  : {homo_energy:.4f} Hartree")
print(f"   LUMO Energy  : {lumo_energy:.4f} Hartree")

# Mulliken Atomic Charges - A first approximation of electron population in the atoms.
print("\nMulliken Atomic Charges:")
charges = mf.mulliken_pop()[1]
for i, q in enumerate(charges):
    symbol = mol.atom_symbol(i)
    print(f"   {symbol:2} atom {i+1}: Charge = {q:+.3f}")

# Orbital Energies - Useful for visualizing energy levels or building orbital diagrams.
print("\nFirst few Molecular Orbital Energies (Hartree):")
for i, e in enumerate(mo_energy[:10]):
    occ = "occ" if i <= homo_index else "vir"
    print(f"   Orbital {i:2d} ({occ}): Energy = {e:.4f} Hartree")
#This is how we can get forces from this calculation
force = mf.nuc_grad_method().kernel()

In [ ]:
##### Lets visualize the Molecular Orbitals that obtained
from pyscf.tools import cubegen
import py3Dmol
import tempfile
import os

# Choose which orbital to visualize
orbital_index = mol.nelectron // 2 - 1  # <---- HOMO
# To view other orbitals, just change orbital_index to any valid index we outputed in the last cell

# To produce a visualization, we will generate a cube file for the selected orbital
tmp_dir = tempfile.mkdtemp()
cube_filename = os.path.join(tmp_dir, f"mo_{orbital_index}.cube")
cubegen.orbital(mol, cube_filename, mf.mo_coeff[:, orbital_index], nx=40)
# Check the use of cubegen in PySCF https://pyscf.org/pyscf_api_docs/pyscf.tools.html#pyscf.tools.cubegen.orbital

# Reads the cube file
with open(cube_filename) as f:
    cube_data = f.read()

# builds an interactive 3D viewer
viewer = py3Dmol.view(width=400, height=400)
viewer.addModel(cube_data, 'cube')  # Load cube file
viewer.setStyle({'stick':{'radius': 0.1},'sphere':{'radius': 0.3}})      # Show atoms and bonds as sticks
viewer.addVolumetricData(cube_data, 'cube', {'isoval': 0.05, 'color': 'blue', 'opacity': 0.7})
viewer.addVolumetricData(cube_data, 'cube', {'isoval': -0.05, 'color': 'red', 'opacity': 0.7})
viewer.zoomTo()
viewer.show()

# Clean up temp files (optional, but important to know how to do, cause cube files can be heavy)
# os.remove(cube_filename)
# os.rmdir(tmp_dir)

# Question: Those orbitals have the shape you would expect?
# https://chem.libretexts.org/Bookshelves/General_Chemistry/General_Chemistry_Supplement_%28Eames%29/Molecular_Orbital_Theory/MO_Diagrams_for_Water_and_Nitrate_Ion

## Computing an IR spectrum

We can also use PySCF and the converged electronic structure calculation to predict the vibrational frequencies of the molecule and construct an approximation to the real IR spectrum. The first step to do so is to compute the **Hessian**, which contains the second derivatives of the energy with respect to the nuclear coordinates. Diagonalizing the mass-weighted Hessian gives the molecular **normal modes** and their vibrational frequencies. For a non-linear molecule with $N$ atoms, we expect $3N - 6$ vibrational modes.

A vibration produces an IR signal only if the molecular dipole moment changes during the vibration. The IR intensity is therefore related to
$$
I_k \propto
\left|
\frac{\partial \boldsymbol{\mu}}
{\partial Q_k}
\right|^2
$$
where $Q_k$ is the normal coordinate of vibrational mode $k$.

PySCF gives us the frequencies and normal modes from the Hessian, but (different from other quantum chemistry codes like ORCA or Gaussian) does not directly provide IR intensities. As a workarround, we can numerically estimate the dipole derivative by slightly displacing the molecule along each normal mode:

$$
\frac{\partial \boldsymbol{\mu}}{\partial Q_k}
\approx
\frac{
\boldsymbol{\mu}(Q_k+\Delta Q)
-
\boldsymbol{\mu}(Q_k-\Delta Q)
}{
2\Delta Q
}
$$

The resulting intensities will therefore be shown as **relative intensities** rather than absolute experimental units. Finally, the discrete vibrational transitions are broadened with Gaussian functions to produce something that looks more like an experimental IR spectrum.

**OBS.:** In a real vibrational-frequency calculation, the molecular geometry should first be optimized. Imaginary frequencies can indicate that the geometry is not at a minimum on the potential-energy surface.

In [ ]:
##### Let's compute the harmonic IR spectrum
import numpy as np
import matplotlib.pyplot as plt
from pyscf.hessian import thermo

# Compute the molecular Hessian
hessian = mf.Hessian().kernel()

# Get the vibrational frequencies and normal modes
vib = thermo.harmonic_analysis(mol, hessian, imaginary_freq=False)

frequencies = np.asarray(vib["freq_wavenumber"]).real
normal_modes = np.asarray(vib["norm_mode"]).real

print("Vibrational Frequencies:")
for i, freq in enumerate(frequencies):
    print(f"   Mode {i+1:2d}: {freq:.2f} cm^-1")


##### Now let's estimate the IR intensities

# Remove translations/rotations or very small frequencies
keep = frequencies > 10
frequencies = frequencies[keep]
normal_modes = normal_modes[keep]

# Original molecular geometry
coords = mol.atom_coords()

# Small displacement along each normal mode
dq = 0.01

# Scanner lets us easily repeat the SCF calculation at nearby geometries
scanner = mf.as_scanner()
scanner.verbose = 0

intensities = []

for mode in normal_modes:

    # Displace the molecule in both directions along this vibration
    mol_plus = mol.set_geom_(coords + dq*mode, unit="Bohr", inplace=False)
    mol_minus = mol.set_geom_(coords - dq*mode, unit="Bohr", inplace=False)

    # Compute the dipole moment at both geometries
    scanner(mol_plus)
    dipole_plus = np.asarray(scanner.dip_moment(unit="Debye", verbose=0))

    scanner(mol_minus)
    dipole_minus = np.asarray(scanner.dip_moment(unit="Debye", verbose=0))

    # Numerical derivative of the dipole along the normal mode
    dipole_derivative = (dipole_plus - dipole_minus) / (2*dq)

    # IR intensity is proportional to |d(mu)/dQ|^2
    intensities.append(np.sum(dipole_derivative**2))

intensities = np.asarray(intensities)

# Normalize so the strongest transition has intensity 100
relative_intensity = 100 * intensities / intensities.max()

print("\nCalculated IR transitions:")
for i, (freq, intensity) in enumerate(zip(frequencies, relative_intensity)):
    print(f"   Mode {i+1:2d}: {freq:8.2f} cm^-1   Intensity = {intensity:6.1f}")


# Let's plot the IR spectrum
# Broaden each vibrational transition with a Gaussian
wavenumbers = np.linspace(0, 4000, 5000)
spectrum = np.zeros_like(wavenumbers)

sigma = 20  # peak width in cm^-1

for freq, intensity in zip(frequencies, relative_intensity):
    spectrum += intensity * np.exp(
        -0.5 * ((wavenumbers - freq) / sigma)**2
    )

spectrum = 100 * spectrum / spectrum.max()

plt.figure(figsize=(9, 5))
plt.plot(wavenumbers, spectrum)

# Also show the calculated transitions as sticks
plt.vlines(frequencies, 0, relative_intensity, alpha=0.3)

plt.xlabel(r"Wavenumber (cm$^{-1}$)")
plt.ylabel("Relative Intensity")
plt.title("Calculated Harmonic IR Spectrum")

# IR spectra are usually plotted with high wavenumbers on the left
plt.gca().invert_xaxis()

plt.show()

## QUESTION: How those values compare with the ones in the liuterature?
# https://en.wikipedia.org/wiki/Electromagnetic_absorption_by_water

## Practice 1

Now, you can try to compute a particular system that you like. Maybe try to run something that has charges and different spin states. Maybe try to define a calculation with another functional? Or a small coupled cluster one? Check https://pyscf.org/quickstart.html or https://pyscf.org/user.html for more informations on how to do it and all the posibilities with PySCF.

Be aware that maybe you will need to play first with low values of mf.conv_tol and mf.max_cycle, to be able to do it in the tutorial time. **Be free to skip this task and finish the notebook before.**  

In [ ]:
from pyscf import dft,gto,scf

#mol = gto.M(
#    atom = '''
#    XX  0.000   0.000   0.000''',
#    basis = '...'
#    )

#mf = ...

#mf.kernel()

#mf.analyze()

# ...

---
---
---

## How to use PySCF for Bulk Systems

While it is primarily designed for molecular quantum chemistry using Gaussian basis sets, PySCF also includes support for periodic (bulk) systems through its `pbc` modules. Unlike plane-wave-based codes (e.g., VASP or Quantum ESPRESSO), PySCF handles periodic calculations using Gaussian basis sets with k-point sampling. This makes it well-suited for prototyping periodic electronic structure methods, but it comes with limitations in terms of scalability and performance for large or complex solids. In this section, we illustrate how to define simple periodic systems (by no particular reason, a [diamond](https://diamond-diadem.github.io/) crystal), perform DFT calculations, generate the band structure, and understand what kinds of bulk simulations you can do with PySCF.

In [ ]:
import pyscf.pbc.tools.pyscf_ase as pyscf_ase
import pyscf.pbc.gto as pbcgto
import pyscf.pbc.dft as pbcdft

import matplotlib.pyplot as plt

from ase.build import bulk


In [ ]:
##### Example of setting up a periodic calculation with PySCF using ASE

from ase.build import bulk
from pyscf.pbc import gto as pbcgto
from pyscf.pbc.tools import pyscf_ase

#Builds a diamond structure of carbon using ASE
c = bulk('C', 'diamond', a=3.5668)       # Cubic diamond lattice with lattice constant a = 3.5668 Å
print("Volume of the simulation cell: ",c.get_volume(), "Å³") # Prints the volume of the unit cell, just to check...
#QUESTION: We have the correct density? Each carbon atom has mass of 12 amu (1.9945×10^-23 g).

#Defines a periodic PySCF cell object using ASE atoms
cell = pbcgto.Cell()
cell.atom = pyscf_ase.ase_atoms_to_pyscf(c)  # Converts ASE atoms to PySCF format
cell.a = c.cell                              # Copies the lattice vectors from the ASE object

cell.basis = 'gth-szv'                   # Basis set: single-zeta valence with Gaussian-type orbitals
cell.pseudo = 'gth-pade'                # Pseudopotential: Goedecker-Teter-Hutter with Pade exchange
#This combination is too simple... Lets use it here just to get some results fast...
cell.verbose = 0                        # Controls the amount of printed output (0 = silent, try changing it and defining a output file)

cell = cell.build(None, None)           # Finalizes and builds the periodic cell object


In [ ]:
##### Generate and visualize a supercell using ASE + py3Dmol
import io
from ase.build import make_supercell, find_optimal_cell_shape

# Finds an optimal 3D integer matrix P such that the supercell has ~256 atoms (just because it makes a nice cube!)
P = find_optimal_cell_shape(c.cell, 256, 'sc')  # 'sc' = simple cubic-like supercell
supercell = make_supercell(c, P)                # Builds the supercell using the transformation matrix P

# Converts the ASE Atoms object to an XYZ format string (in-memory) for visualization
xyz_string = io.StringIO()
supercell.write(xyz_string, format='xyz')       # Writes atomic coordinates in XYZ format to the string buffer

# Creates an interactive 3D viewer with py3Dmol
view = py3Dmol.view(width=600, height=400)

# Loads the atomic structure from the XYZ string into the viewer
view.addModel(xyz_string.getvalue(), 'xyz')

# Sets visualization style: sticks and spheres (ball-and-stick)
view.setStyle({'stick': {'radius': 0.2}, 'sphere': {'radius': 0.5}})

# Automatically zooms to fit the whole structure in the view
view.zoomTo()

# Displays the 3D visualization
view.show()


In [ ]:
##### Define a high-symmetry k-point path for band structure calculation

from ase.dft.kpoints import sc_special_points as special_points, get_bandpath

# Retrieves high-symmetry points for the face-centered cubic (FCC) Brillouin zone
points = special_points['fcc']
G = points['G']     # Gamma point (center of Brillouin zone)
X = points['X']
W = points['W']
K = points['K']
L = points['L']

# Defines a k-point path through high-symmetry points in the Brillouin zone
# Path: L → Γ → X → W → K → Γ
path = get_bandpath([L, G, X, W, K, G], c.cell, npoints=50)  # 50 points between each segment

# Converts the relative k-points to absolute Cartesian coordinates (Bohr⁻¹)
band_kpts = cell.get_abs_kpts(path.kpts)

# Generates x-axis values and labels for plotting (e.g., Γ-X-W...)
x, X, sp_points = path.get_linear_kpoint_axis()


In [ ]:
##### Perform a DFT single-point energy calculation and compute band structure

import time
from pyscf.pbc import dft as pbcdft

# Start timer for SCF calculation
t0 = time.time()

# Define and run a periodic DFT calculation (RKS = spin-restricted Kohn-Sham)
mf = pbcdft.RKS(cell)
print(mf.kernel())                    # Runs the SCF and prints the total energy

# Print timing for SCF step
tf = time.time()
print(f"Time on the Single Point calculation: {tf - t0:.2f} seconds")

# Start timer for band structure calculation
t0 = time.time()

# Computes the band energies at the specified k-points along the high-symmetry path
e_kn = mf.get_bands(band_kpts)[0]     # e_kn is a list of arrays: one band structure per k-point

# Normalize band energies so that the valence band maximum (VBM) is set to 0 eV
vbmax = -99
for en in e_kn:
    vb_k = en[cell.nelectron // 2 - 1]  # Energy of the highest occupied band at each k-point
    if vb_k > vbmax:
        vbmax = vb_k
e_kn = [en - vbmax for en in e_kn]     # Shift all bands by the VBM

# Print timing for band structure step
tf = time.time()
print(f"Time on the Band Structure calculation: {tf - t0:.2f} seconds")


In [ ]:
##### Band structure using a 2×2×2 k-point grid for SCF sampling

import time
from pyscf.pbc import dft as pbcdft

t0 = time.time()
kmf = pbcdft.KRKS(cell, cell.make_kpts([2, 2, 2]))  # <--- 2×2×2 k-point grid. The only thing changing!!!
print(kmf.kernel())
tf = time.time()
print(f"Time on the Single Point calculation: {tf - t0:.2f} seconds")

t0 = time.time()
e_kn_2 = kmf.get_bands(band_kpts)[0]
vbmax = -99
for en in e_kn_2:
    vb_k = en[cell.nelectron // 2 - 1]
    if vb_k > vbmax:
        vbmax = vb_k
e_kn_2 = [en - vbmax for en in e_kn_2]
tf = time.time()
print(f"Time on the Band Structure calculation: {tf - t0:.2f} seconds")


In [ ]:
##### Plot band structures computed from Γ-only and 2×2×2 k-point sampled SCF calculations

import matplotlib.pyplot as plt

au2ev = 27.21139 # Conversion factor: Hartree to electron volts
emin = -1 * au2ev # Energy range for the y-axis (in eV)
emax =  1 * au2ev

plt.figure(figsize=(5, 6))
nbands = cell.nao_nr()  # Number of molecular orbitals (bands) to plot

# Plot first band (n = 0) for both calculations, we are doing it first to set the plot with labels
plt.plot(x, [e[0] * au2ev for e in e_kn],   color='b', label='K = [1,1,1]')   # Γ-point only SCF
plt.plot(x, [e[0] * au2ev for e in e_kn_2], color='r', label='K = [2,2,2]')   # 2×2×2 SCF
# Plot remaining bands (without labels)
for n in range(1, nbands):
    plt.plot(x, [e[n] * au2ev for e in e_kn],   color='b')   # Γ-only bands
    plt.plot(x, [e[n] * au2ev for e in e_kn_2], color='r')   # 2×2×2 bands

# Set x-axis tick labels with symbols for symmetry points
plt.xticks(X, ['$%s$' % n for n in ['L', r'\Gamma', 'X', 'W', 'K', r'\Gamma']])
# Draw vertical lines at each high-symmetry k-point
for p in X:
    plt.plot([p, p], [emin, emax], 'k-')

# Draw a horizontal dotted line at 0 eV (valence band maximum)
plt.plot([0, X[-1]], [0, 0], 'k--')

plt.axis(xmin=0, xmax=X[-1], ymin=emin, ymax=emax)
plt.xlabel('k-vector')
plt.ylabel('Energy (eV)')
plt.legend()
plt.grid(True)
plt.show()

## Practice 2

Now that you have an example of how to get the band structure for diamond, try to modify it for another material. Classical/emblematic nanostructures like graphene or coppers are easy to find online. Try to build the cell from the public information about the material, vizualize the structure and compute the band structure.

Pay attention to definitions like basis set and the reciprobac space symmetry points, those need to be adjusted top the system in question. You can also try different DFT functionals (we didnt definef it in the example, but the default value for PBC in PySCF is `xc='LDA,VWN'`).